# Notebook 7 — Actualizar método manual a corregido
## Celda 1 — Imports y explicación

In [1]:
# =========================
# NOTEBOOK 7
# Actualización de método:
# manual -> corregido
# =========================
#
# Objetivo:
# Actualizar los archivos revisados para que las entidades que fueron corregidas
# durante la revisión manual tengan metodo = "corregido" en lugar de "manual".
#
# Criterio:
# - "corregido": la entidad fue detectada por algún método automático, pero su valor fue corregido.
# - "manual": se reserva para entidades agregadas desde cero que ningún método había detectado.

import json
from pathlib import Path

import pandas as pd

# Celda 2 — Subir archivos

In [2]:
from google.colab import files

uploaded = files.upload()

print("Archivos subidos:")
for filename in uploaded.keys():
    print("-", filename)

Saving embargos_revision_entidades_actualizado.csv to embargos_revision_entidades_actualizado.csv
Saving embargos_revision_entidades_actualizado.json to embargos_revision_entidades_actualizado.json
Saving embargos_revision_entidades_actualizado.xlsx to embargos_revision_entidades_actualizado.xlsx
Archivos subidos:
- embargos_revision_entidades_actualizado.csv
- embargos_revision_entidades_actualizado.json
- embargos_revision_entidades_actualizado.xlsx


# Celda 3 — Detectar rutas automáticamente

In [3]:
# =========================
# DETECCIÓN AUTOMÁTICA DE ARCHIVOS
# =========================

uploaded_files = list(uploaded.keys())

json_files = [f for f in uploaded_files if f.lower().endswith(".json")]
csv_files = [f for f in uploaded_files if f.lower().endswith(".csv")]
xlsx_files = [f for f in uploaded_files if f.lower().endswith(".xlsx")]

if len(json_files) != 1:
    raise ValueError(f"Se esperaba 1 archivo JSON, pero se encontraron: {json_files}")

if len(csv_files) != 1:
    raise ValueError(f"Se esperaba 1 archivo CSV, pero se encontraron: {csv_files}")

if len(xlsx_files) != 1:
    raise ValueError(f"Se esperaba 1 archivo XLSX, pero se encontraron: {xlsx_files}")

JSON_PATH = Path(json_files[0])
CSV_PATH = Path(csv_files[0])
EXCEL_PATH = Path(xlsx_files[0])

print("JSON detectado:", JSON_PATH)
print("CSV detectado:", CSV_PATH)
print("Excel detectado:", EXCEL_PATH)

JSON detectado: embargos_revision_entidades_actualizado.json
CSV detectado: embargos_revision_entidades_actualizado.csv
Excel detectado: embargos_revision_entidades_actualizado.xlsx


# Celda 4 — Configurar nombres de salida

In [4]:
# =========================
# ARCHIVOS DE SALIDA
# =========================

OUTPUT_JSON = Path("/content/embargos_revision_entidades_actualizado_metodo_corregido.json")
OUTPUT_CSV = Path("/content/embargos_revision_entidades_actualizado_metodo_corregido.csv")
OUTPUT_EXCEL = Path("/content/embargos_revision_entidades_actualizado_metodo_corregido.xlsx")

print("Salida JSON:", OUTPUT_JSON)
print("Salida CSV:", OUTPUT_CSV)
print("Salida Excel:", OUTPUT_EXCEL)

Salida JSON: /content/embargos_revision_entidades_actualizado_metodo_corregido.json
Salida CSV: /content/embargos_revision_entidades_actualizado_metodo_corregido.csv
Salida Excel: /content/embargos_revision_entidades_actualizado_metodo_corregido.xlsx


# Celda 5 — Función para normalizar método

In [5]:
def normalizar_metodo(valor):
    """
    Reemplaza el método 'manual' por 'corregido'.

    También contempla variantes por si hay mayúsculas, espacios o tildes.
    """
    if pd.isna(valor):
        return valor

    metodo = str(valor).strip()

    variantes_manual = {
        "manual",
        "Manual",
        "MANUAL",
        "revision manual",
        "revisión manual",
        "Revision manual",
        "Revisión manual",
        "REVISION MANUAL",
        "REVISIÓN MANUAL",
    }

    if metodo in variantes_manual or metodo.lower() in {"manual", "revision manual", "revisión manual"}:
        return "corregido"

    return metodo

# Celda 6 — Actualizar JSON

In [6]:
# =========================
# ACTUALIZAR JSON
# =========================

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data_json = json.load(f)

contador_json = 0

for doc in data_json:
    for ent in doc.get("entidades", []):
        metodo_original = ent.get("metodo")

        metodo_nuevo = normalizar_metodo(metodo_original)

        if metodo_original != metodo_nuevo:
            contador_json += 1
            ent["metodo"] = metodo_nuevo

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data_json, f, ensure_ascii=False, indent=2)

print("JSON actualizado guardado en:", OUTPUT_JSON)
print("Reemplazos realizados en JSON:", contador_json)

JSON actualizado guardado en: /content/embargos_revision_entidades_actualizado_metodo_corregido.json
Reemplazos realizados en JSON: 105


# Celda 7 — Actualizar CSV

In [7]:
# =========================
# ACTUALIZAR CSV
# =========================

df_csv = pd.read_csv(CSV_PATH)

if "metodo" not in df_csv.columns:
    raise ValueError("El CSV no tiene columna 'metodo'.")

contador_csv = (
    df_csv["metodo"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["manual", "revision manual", "revisión manual"])
    .sum()
)

df_csv["metodo"] = df_csv["metodo"].apply(normalizar_metodo)

df_csv.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("CSV actualizado guardado en:", OUTPUT_CSV)
print("Reemplazos realizados en CSV:", contador_csv)

CSV actualizado guardado en: /content/embargos_revision_entidades_actualizado_metodo_corregido.csv
Reemplazos realizados en CSV: 105


# Celda 8 — Actualizar Excel

Esta celda actualiza todas las hojas que tengan columna metodo.

In [8]:
# =========================
# ACTUALIZAR EXCEL
# =========================

excel = pd.ExcelFile(EXCEL_PATH)

print("Hojas encontradas:")
print(excel.sheet_names)

contador_excel_total = 0
sheets_actualizadas = {}

for sheet_name in excel.sheet_names:
    df_sheet = pd.read_excel(EXCEL_PATH, sheet_name=sheet_name)

    if "metodo" in df_sheet.columns:
        contador_sheet = (
            df_sheet["metodo"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(["manual", "revision manual", "revisión manual"])
            .sum()
        )

        df_sheet["metodo"] = df_sheet["metodo"].apply(normalizar_metodo)
        contador_excel_total += contador_sheet

        print(f"Hoja actualizada: {sheet_name} | Reemplazos: {contador_sheet}")
    else:
        print(f"Hoja sin columna metodo, se conserva igual: {sheet_name}")

    sheets_actualizadas[sheet_name] = df_sheet

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    for sheet_name, df_sheet in sheets_actualizadas.items():
        # Excel permite máximo 31 caracteres en el nombre de hoja
        safe_sheet_name = sheet_name[:31]
        df_sheet.to_excel(writer, sheet_name=safe_sheet_name, index=False)

print("Excel actualizado guardado en:", OUTPUT_EXCEL)
print("Reemplazos realizados en Excel:", contador_excel_total)

Hojas encontradas:
['Revision_Actualizada', 'Resumen', 'Instrucciones']
Hoja actualizada: Revision_Actualizada | Reemplazos: 105
Hoja sin columna metodo, se conserva igual: Resumen
Hoja sin columna metodo, se conserva igual: Instrucciones
Excel actualizado guardado en: /content/embargos_revision_entidades_actualizado_metodo_corregido.xlsx
Reemplazos realizados en Excel: 105


# Celda 9 — Validaciones finales

In [9]:
# =========================
# VALIDACIONES FINALES
# =========================

print("Validación CSV:")
print(df_csv["metodo"].value_counts(dropna=False))

manuales_csv = (
    df_csv["metodo"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["manual", "revision manual", "revisión manual"])
    .sum()
)

print("\nValores manual restantes en CSV:", manuales_csv)


print("\nValidación JSON:")

metodos_json = []

for doc in data_json:
    for ent in doc.get("entidades", []):
        metodos_json.append(ent.get("metodo"))

metodos_json_series = pd.Series(metodos_json)

print(metodos_json_series.value_counts(dropna=False))

manuales_json = (
    metodos_json_series
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["manual", "revision manual", "revisión manual"])
    .sum()
)

print("\nValores manual restantes en JSON:", manuales_json)

Validación CSV:
metodo
gliner_fastino             915
regex                      336
regex_contextual           255
corregido                  105
regex_persona_cerca_dni     15
Name: count, dtype: int64

Valores manual restantes en CSV: 0

Validación JSON:
gliner_fastino             911
regex                      336
regex_contextual           255
corregido                  105
regex_persona_cerca_dni     15
Name: count, dtype: int64

Valores manual restantes en JSON: 0


# Celda 10 — Descargar archivos actualizados

In [10]:
from google.colab import files

files.download(str(OUTPUT_JSON))
files.download(str(OUTPUT_CSV))
files.download(str(OUTPUT_EXCEL))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Objetivo del notebook

Este notebook actualiza los archivos de entidades revisadas para corregir el nombre del método utilizado en las entidades modificadas durante la revisión.

Originalmente, las entidades corregidas fueron marcadas con el método `manual`. Sin embargo, se decidió que el nombre correcto del método debe ser `corregido`, porque esas entidades no fueron agregadas desde cero manualmente, sino que fueron extraídas automáticamente y luego corregidas durante la revisión.

El método `manual` queda reservado para futuras entidades que sean agregadas completamente a mano porque ningún método automático las detectó.

Archivos de entrada:
- `embargos_revision_entidades_actualizado.json`
- `embargos_revision_entidades_actualizado.csv`
- `embargos_revision_entidades_actualizado.xlsx`

Archivos de salida:
- `embargos_revision_entidades_actualizado_metodo_corregido.json`
- `embargos_revision_entidades_actualizado_metodo_corregido.csv`
- `embargos_revision_entidades_actualizado_metodo_corregido.xlsx`